# Cross-Lingual Abstractive Text Summarization: mT5 vs mBART
In this notebook, we will implement and compare two multilingual sequence-to-sequence models (**mT5** and **mBART**) for abstractive text summarization on the **Bengali** language subset of the **XL-Sum** dataset.

### Metrics for Comparison:
- **ROUGE-1, ROUGE-2, ROUGE-L**
- **mBERT Score** (F1, Precision, Recall)

In [ ]:
!pip install -q datasets transformers[torch] evaluate rouge_score sentencepiece accelerate bert_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from datasets import load_dataset

# Loading the pre-converted parquet files directly from the Hugging Face hub mirror for Bengali XL-Sum
dataset = load_dataset(
    "parquet",
    data_files="https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/refs%2Fconvert%2Fparquet/bengali/test/0000.parquet",
    split="train[:10]"
)

print(f"Loaded {len(dataset)} examples for evaluation.")
# Displaying the first sample's keys and preview
print("Keys available:", dataset.column_names)
print("\nSample Text:", dataset[0]['text'][:300] + "...")
print("\nSample Summary:", dataset[0]['summary'])

bengali/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.51MB            

bengali/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 10 examples for evaluation.
Keys available: ['id', 'url', 'title', 'summary', 'text']

Sample Text: আটক স্বজনদের আইডি কার্ড হাতে উইগুর নারীদের বিক্ষোভ (ফাইল ফটো) অভিযোগ করেন জাতিসংঘের জাতিগত বৈষম্য বিষয়ক কমিটির সদস্য গে ম্যাকডুগাল। তিনি বলেন, এমন খবর তিনি পেয়েছেন যে চীনা কর্তৃপক্ষ স্বায়ত্তশাসিত উইগুর প্রদেশকে কার্যত "বিশাল একটি বন্দীশিবিরে" রূপান্তরিত করেছে। চীন সাথে সাথে এর কোনো জবাব দেয়নি। ৫...

Sample Summary: জাতিসংঘের মানবাধিকার বিষয়ক কমিটি জেনেভাতে চীনের ওপর দুদিনের যে বিশেষ সভা করছে, সেখানে অভিযোগ করা হয়, কট্টরপন্থী সন্দেহে ১০ লাখ চীনা উইগুর মুসলিমকে বিভিন্ন রাজনৈতিক শিবিরে আটকে রাখা হয়েছে।


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import pandas as pd
import numpy as np

# Initialize evaluation metrics
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# 1. Evaluation setup for mT5
mt5_model_name = "csebuetnlp/mT5_multilingual_XLSum"  # Fine-tuned on XL-Sum
print(f"Loading mT5 tokenizer and model: {mt5_model_name}...")
mt5_tokenizer = AutoTokenizer.from_pretrained(mt5_model_name, legacy=False)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_model_name).to(device)

def summarize_mt5(text):
    inputs = mt5_tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
    summary_ids = mt5_model.generate(
        inputs["input_ids"],
        max_length=84,
        num_beams=4,
        length_penalty=1.0,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    return mt5_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

Loading mT5 tokenizer and model: csebuetnlp/mT5_multilingual_XLSum...


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
# 2. Evaluation setup for mBART using the official public Meta multi-lingual model
mbart_model_name = "facebook/mbart-large-50-many-to-many-mmt"
print(f"Loading mBART tokenizer and model: {mbart_model_name}...")
mbart_tokenizer = AutoTokenizer.from_pretrained(mbart_model_name)
mbart_model = AutoModelForSeq2SeqLM.from_pretrained(mbart_model_name).to(device)

# Setting the tokenizer target language to Bengali
mbart_tokenizer.src_lang = "bn_IN"

def summarize_mbart(text):
    inputs = mbart_tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
    summary_ids = mbart_model.generate(
        inputs["input_ids"],
        forced_bos_token_id=mbart_tokenizer.lang_code_to_id["bn_IN"],
        max_length=84,
        num_beams=4,
        length_penalty=1.0,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    return mbart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

Loading mBART tokenizer and model: facebook/mbart-large-50-many-to-many-mmt...


config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [ ]:
# 3. Run predictions
references = [item['summary'] for item in dataset]
texts = [item['text'] for item in dataset]

mt5_predictions = []
mbart_predictions = []

print("Generating summaries...")
for idx, text in enumerate(texts):
    print(f"Processing sample {idx+1}/{len(texts)}...")
    mt5_predictions.append(summarize_mt5(text))
    mbart_predictions.append(summarize_mbart(text))

print("Generation completed!")

Generating summaries...
Processing sample 1/10...
Processing sample 2/10...
Processing sample 3/10...
Processing sample 4/10...
Processing sample 5/10...
Processing sample 6/10...
Processing sample 7/10...
Processing sample 8/10...
Processing sample 9/10...
Processing sample 10/10...
Generation completed!


In [ ]:
# 4. Compute Metrics
def compute_metrics(predictions, references):
    # Compute ROUGE
    rouge_results = rouge.compute(predictions=predictions, references=references, use_stemmer=True)

    # Compute mBERT score (using bertscore with multilingual BERT)
    bert_results = bertscore.compute(
        predictions=predictions,
        references=references,
        model_type="bert-base-multilingual-cased",
        lang="bn"
    )

    return {
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "mbert_precision": np.mean(bert_results["precision"]),
        "mbert_recall": np.mean(bert_results["recall"]),
        "mbert_f1": np.mean(bert_results["f1"])
    }

print("Evaluating mT5...")
mt5_metrics = compute_metrics(mt5_predictions, references)

print("Evaluating mBART...")
mbart_metrics = compute_metrics(mbart_predictions, references)

# Present evaluation results
results_df = pd.DataFrame([mt5_metrics, mbart_metrics], index=["mT5", "mBART"])
print("\n--- Performance Comparison Table ---")
print(results_df.round(4))

Evaluating mT5...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating mBART...

--- Performance Comparison Table ---
       rouge1  rouge2  rougeL  mbert_precision  mbert_recall  mbert_f1
mT5       0.0     0.0     0.0           0.8051        0.7648    0.7841
mBART     0.0     0.0     0.0           0.6080        0.6029    0.6054
